# FPS Paper Results

This notebook generates the FPS-version result artifacts from the checked-in PBS analysis files. It focuses on PBS cost, accuracy, and compact LaTeX tables. It does not compute or report EI-vs-Boolean time reduction/speedup claims.

Generated files are written to `results/fps_paper_results/`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "artifacts" / "measurements").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "artifacts" / "measurements"
OUT_DIR = PROJECT_ROOT / "build" / "notebook-output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Main FPS PBS-cost table includes absolute EI_DDLGN evaluation time from the
# current boolean_vs_ei comparison CSV. No speedup or time-reduction column is reported.
INCLUDE_EI_EVAL_TIME = True

# Keep this True if you want a compact MNIST comparison table against the
# reproduced QAT-FCNN arithmetic baselines from the previous manuscript.
GENERATE_ARITHMETIC_COMPARISON = True

PAPER_TITLE_FONTSIZE = 20
PAPER_LABEL_FONTSIZE = 17
PAPER_TICK_FONTSIZE = 15
PAPER_LEGEND_FONTSIZE = 12
PAPER_LEGEND_TITLE_FONTSIZE = 13
PAPER_HEATMAP_ANNOTATION_FONTSIZE = 14
PAPER_SPINE_WIDTH = 1.4
PAPER_LINEWIDTH = 2.8
PAPER_MARKERSIZE = 7

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "font.size": PAPER_TICK_FONTSIZE,
    "font.weight": "bold",
    "axes.labelsize": PAPER_LABEL_FONTSIZE,
    "axes.labelweight": "bold",
    "axes.titlesize": PAPER_TITLE_FONTSIZE,
    "axes.titleweight": "bold",
    "axes.grid": True,
    "grid.alpha": 0.25,
    "xtick.labelsize": PAPER_TICK_FONTSIZE,
    "ytick.labelsize": PAPER_TICK_FONTSIZE,
    "legend.fontsize": PAPER_LEGEND_FONTSIZE,
    "legend.title_fontsize": PAPER_LEGEND_TITLE_FONTSIZE,
})

def apply_paper_axis_style(ax):
    ax.tick_params(axis="both", labelsize=PAPER_TICK_FONTSIZE, width=PAPER_SPINE_WIDTH, length=5)
    for tick in ax.get_xticklabels() + ax.get_yticklabels():
        tick.set_fontweight("bold")
    for spine in ax.spines.values():
        spine.set_linewidth(PAPER_SPINE_WIDTH)

def apply_paper_legend_style(legend):
    if legend is None:
        return
    for text in legend.get_texts():
        text.set_fontsize(PAPER_LEGEND_FONTSIZE)
        text.set_fontweight("bold")
    title = legend.get_title()
    if title is not None:
        title.set_fontsize(PAPER_LEGEND_TITLE_FONTSIZE)
        title.set_fontweight("bold")
    legend.get_frame().set_linewidth(PAPER_SPINE_WIDTH)

DATASET_ORDER = ["MNIST", "FashionMNIST", "UCI Phishing Websites"]
DATASET_LABELS = {
    "MNIST": "MNIST",
    "FashionMNIST": "FashionMNIST",
    "UCI Phishing Websites": "UCI Phishing",
}
COLORS = {
    "MNIST": "#1f77b4",
    "FashionMNIST": "#ff7f0e",
    "UCI Phishing Websites": "#2ca02c",
}

print(f"Project root: {PROJECT_ROOT}")
print(f"Output dir:   {OUT_DIR}")

## Load Result Tables

In [ ]:
selected_models = pd.read_csv(DATA_DIR / "selected_models.csv")
all_models = pd.read_csv(DATA_DIR / "pbs_summary.csv")
dataset_summary = pd.read_csv(DATA_DIR / "pbs_dataset_summary.csv")
ei_timing = pd.read_csv(DATA_DIR / "encrypted_timings.csv")

def width_to_int(value):
    text = str(value).strip().upper()
    if text.endswith("K"):
        return int(float(text[:-1]) * 1000)
    return int(float(text))

selected_models["Width"] = selected_models["W"].map(width_to_int)
selected_models["Model Size"] = pd.Categorical(
    selected_models["Model Size"],
    categories=["Small", "Medium", "Large"],
    ordered=True,
)
selected_models["Dataset"] = pd.Categorical(
    selected_models["Dataset"],
    categories=DATASET_ORDER,
    ordered=True,
)
selected_models = selected_models.sort_values(["Model Size", "Dataset"]).reset_index(drop=True)

timing_key = ei_timing.rename(columns={
    "dataset_name": "Dataset",
    "depth": "D",
    "width": "Width",
})[["Dataset", "D", "Width", "samples", "ei_avg_eval_seconds", "prediction_match_rate"]]

selected_with_timing = selected_models.merge(timing_key, on=["Dataset", "D", "Width"], how="left")

all_models["Optimized PBS (k)"] = all_models["Optimized PBS"] / 1000.0
all_models["Baseline PBS (k)"] = all_models["Baseline PBS"] / 1000.0
all_models["Dataset"] = pd.Categorical(all_models["Dataset"], categories=DATASET_ORDER, ordered=True)
all_models = all_models.sort_values(["Dataset", "Optimized PBS", "Test Accuracy %"]).reset_index(drop=True)

display(selected_with_timing)
display(dataset_summary)

## Selected EI-DDLGN PBS Table

This table is intended for the main FPS paper. It includes absolute EI-DDLGN evaluation time from `encrypted_timings.csv`, but no speedup or time-reduction column.

In [ ]:
def make_selected_table(include_timing=False):
    df = selected_with_timing.copy()
    df["Model"] = df.apply(lambda row: f"{row['Model Size']} (${int(row['D'])}\\times {row['W']}$)", axis=1)
    df["Dataset"] = df["Dataset"].astype(str).map(DATASET_LABELS)
    keep = [
        "Model",
        "Dataset",
        "Accuracy (%)",
        "Actual Constant (%)",
        "PBS Before Optimization (%)",
        "PBS After Optimization (%)",
        "PBS Reduction (%)",
    ]
    if include_timing:
        keep.append("ei_avg_eval_seconds")
    df = df[keep].rename(columns={
        "Accuracy (%)": "Acc. (\\%)",
        "Actual Constant (%)": "Public const. (\\%)",
        "PBS Before Optimization (%)": "PBS before (\\%)",
        "PBS After Optimization (%)": "PBS after (\\%)",
        "PBS Reduction (%)": "PBS red. (\\%)",
        "ei_avg_eval_seconds": "EI eval. (s)",
    })
    numeric_cols = [c for c in df.columns if c not in {"Model", "Dataset"}]
    df[numeric_cols] = df[numeric_cols].round(2)
    return df

def export_selected_latex(df, output_path, caption, label):
    column_format = "ll" + "r" * (df.shape[1] - 2)
    latex = df.set_index(["Model", "Dataset"]).to_latex(
        escape=False,
        multirow=True,
        index_names=False,
        float_format="%.2f",
        caption=caption,
        label=label,
        column_format=column_format,
        position="t",
    )
    latex = latex.replace("\\begin{table}[t]\n", "\\begin{table}[t]\n\\centering\n")
    latex = latex.replace(" &  & ", "Model & Dataset & ", 1)
    latex = latex.replace("\\begin{tabular}", "\\scriptsize\n\\setlength{\\tabcolsep}{3pt}\n\\begin{tabular}")
    output_path.write_text(latex, encoding="utf-8")
    return latex

selected_no_timing = make_selected_table(include_timing=False)
selected_with_absolute_time = make_selected_table(include_timing=True)

selected_with_absolute_time.to_csv(OUT_DIR / "fps_selected_models_table.csv", index=False)
selected_no_timing.to_csv(OUT_DIR / "fps_selected_models_table_without_ei_eval_time.csv", index=False)

latex_with_time = export_selected_latex(
    selected_with_absolute_time,
    OUT_DIR / "fps_selected_models_table.tex",
    "Selected EI-DDLGN models across datasets. PBS before/after are reported as the percentage of model gates requiring bootstrapping before and after public-constant propagation. EI evaluation time is the absolute measured evaluation time per sample.",
    "tab:fps-selected-ei-ddlgn-pbs",
)

latex_no_timing = export_selected_latex(
    selected_no_timing,
    OUT_DIR / "fps_selected_models_table_without_ei_eval_time.tex",
    "Selected EI-DDLGN models across datasets. PBS before/after are reported as the percentage of model gates requiring bootstrapping before and after public-constant propagation.",
    "tab:fps-selected-ei-ddlgn-pbs-without-time",
)

display(selected_with_absolute_time if INCLUDE_EI_EVAL_TIME else selected_no_timing)
print(OUT_DIR / "fps_selected_models_table.tex")

## Figure 1: Accuracy vs Optimized EI-DDLGN PBS Count

In [ ]:
def pareto_frontier(df, x_col, y_col):
    ordered = df.sort_values([x_col, y_col], ascending=[True, True])
    rows = []
    best_y = -np.inf
    for _, row in ordered.iterrows():
        if row[y_col] > best_y + 1e-12:
            rows.append(row)
            best_y = row[y_col]
    return pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(9.2, 5.6))
frontiers = []
for dataset in DATASET_ORDER:
    group = all_models[all_models["Dataset"].astype(str) == dataset]
    ax.scatter(
        group["Optimized PBS (k)"],
        group["Test Accuracy %"],
        s=70,
        alpha=0.68,
        color=COLORS[dataset],
        label=f"{DATASET_LABELS[dataset]} models",
    )
    frontier = pareto_frontier(group, "Optimized PBS (k)", "Test Accuracy %")
    frontiers.append(frontier.assign(dataset_name=dataset))
    ax.plot(
        frontier["Optimized PBS (k)"],
        frontier["Test Accuracy %"],
        marker="o",
        markersize=PAPER_MARKERSIZE,
        linewidth=PAPER_LINEWIDTH,
        color=COLORS[dataset],
        label=f"{DATASET_LABELS[dataset]} Pareto",
    )

ax.set_xlabel("Executed PBS count per inference (thousands)", fontsize=PAPER_LABEL_FONTSIZE, fontweight="bold", labelpad=8)
ax.set_ylabel("Test accuracy (%)", fontsize=PAPER_LABEL_FONTSIZE, fontweight="bold", labelpad=8)
#ax.set_title("Accuracy vs EI-DDLGN Optimized PBS Count")
apply_paper_axis_style(ax)
legend = ax.legend(ncol=2, frameon=True, loc="upper center", bbox_to_anchor=(0.5, 1.24), columnspacing=1.2, handlelength=2.1)
apply_paper_legend_style(legend)
fig.tight_layout()

for ext in ["pdf", "png"]:
    fig.savefig(OUT_DIR / f"fps_accuracy_vs_optimized_pbs.{ext}", bbox_inches="tight")
plt.show()

pareto_df = pd.concat(frontiers, ignore_index=True)
pareto_df.to_csv(OUT_DIR / "fps_accuracy_vs_optimized_pbs_pareto_points.csv", index=False)
print(OUT_DIR / "fps_accuracy_vs_optimized_pbs.pdf")

## Figure 2: Distribution of PBS Reduction Across All Analyzed Models

In [ ]:
fig, ax = plt.subplots(figsize=(8.8, 5.4))
box_data = [
    all_models[all_models["Dataset"].astype(str) == dataset]["PBS Reduction %"].dropna().to_numpy()
    for dataset in DATASET_ORDER
]
positions = np.arange(1, len(DATASET_ORDER) + 1)
box = ax.boxplot(
    box_data,
    positions=positions,
    widths=0.55,
    patch_artist=True,
    showmeans=True,
    boxprops={"linewidth": PAPER_SPINE_WIDTH},
    whiskerprops={"linewidth": PAPER_SPINE_WIDTH},
    capprops={"linewidth": PAPER_SPINE_WIDTH},
    medianprops={"linewidth": 2.2, "color": "black"},
    meanprops={"marker": "D", "markerfacecolor": "white", "markeredgecolor": "black", "markersize": 7, "markeredgewidth": PAPER_SPINE_WIDTH},
    flierprops={"markersize": 6, "markeredgewidth": PAPER_SPINE_WIDTH},
)
for patch, dataset in zip(box["boxes"], DATASET_ORDER):
    patch.set_facecolor(COLORS[dataset])
    patch.set_alpha(0.35)
    patch.set_edgecolor(COLORS[dataset])

rng = np.random.default_rng(7)
for pos, dataset, values in zip(positions, DATASET_ORDER, box_data):
    jitter = rng.normal(0, 0.045, size=len(values))
    ax.scatter(np.full(len(values), pos) + jitter, values, s=52, color=COLORS[dataset], alpha=0.78)

ax.set_xticks(positions)
ax.set_xticklabels([DATASET_LABELS[d] for d in DATASET_ORDER], fontsize=PAPER_TICK_FONTSIZE, fontweight="bold")
ax.set_ylabel("PBS bypass rate (%)", fontsize=PAPER_LABEL_FONTSIZE, fontweight="bold", labelpad=8)
#ax.set_title("PBS Reduction Distribution Across 72 Analyzed Models")
apply_paper_axis_style(ax)
fig.tight_layout()

for ext in ["pdf", "png"]:
    fig.savefig(OUT_DIR / f"fps_pbs_reduction_distribution.{ext}", bbox_inches="tight")
plt.show()

distribution_summary = all_models.groupby(all_models["Dataset"].astype(str), observed=False)["PBS Reduction %"].agg(
    models="count",
    mean="mean",
    median="median",
    min="min",
    max="max",
).loc[DATASET_ORDER].round(3)
distribution_summary.to_csv(OUT_DIR / "fps_pbs_reduction_distribution_summary.csv")
display(distribution_summary)
print(OUT_DIR / "fps_pbs_reduction_distribution.pdf")

## Figure 3: Best Accuracy Under Optimized PBS Budget

In [ ]:
budget_rows = []
for dataset in DATASET_ORDER:
    group = all_models[all_models["Dataset"].astype(str) == dataset].sort_values("Optimized PBS")
    running_best = -np.inf
    for _, row in group.iterrows():
        running_best = max(running_best, row["Test Accuracy %"])
        budget_rows.append({
            "Dataset": dataset,
            "Optimized PBS": row["Optimized PBS"],
            "Optimized PBS (k)": row["Optimized PBS"] / 1000.0,
            "Best accuracy under budget (%)": running_best,
        })

budget_df = pd.DataFrame(budget_rows)
budget_df.to_csv(OUT_DIR / "fps_best_accuracy_under_optimized_pbs_budget.csv", index=False)

fig, ax = plt.subplots(figsize=(9.4, 5.8))
for dataset in DATASET_ORDER:
    group = budget_df[budget_df["Dataset"] == dataset]
    ax.step(
        group["Optimized PBS (k)"],
        group["Best accuracy under budget (%)"],
        where="post",
        linewidth=PAPER_LINEWIDTH,
        marker="o",
        markersize=PAPER_MARKERSIZE,
        color=COLORS[dataset],
        label=DATASET_LABELS[dataset],
    )

#ax.set_title("Best Accuracy Under EI_DDLGN PBS Budget")
ax.set_xlabel("Executed PBS budget\n(thousands/vector)", fontsize=PAPER_LABEL_FONTSIZE, fontweight="bold", labelpad=8)
ax.set_ylabel("Best test accuracy (%)", fontsize=PAPER_LABEL_FONTSIZE, fontweight="bold", labelpad=8)
ax.grid(True, linestyle="--", alpha=0.3)
apply_paper_axis_style(ax)
legend = ax.legend(loc="lower right", frameon=True)
apply_paper_legend_style(legend)
fig.tight_layout()

for ext in ["pdf", "png"]:
    fig.savefig(OUT_DIR / f"fps_best_accuracy_under_optimized_pbs_budget.{ext}", bbox_inches="tight")
plt.show()
print(OUT_DIR / "fps_best_accuracy_under_optimized_pbs_budget.pdf")

## Figure 4: Optimized PBS Gate Share Heatmaps

This heatmap reports the percentage of gates that still require PBS after applying the public-constant PBS reduction trick: `100 * Optimized PBS / Total Gates`.

In [ ]:
pbs_gate_share = all_models.copy()
pbs_gate_share["PBS Gate Share After Reduction (%)"] = (
    100.0 * pbs_gate_share["Optimized PBS"] / pbs_gate_share["Total Gates"]
)

pbs_gate_share_source = pbs_gate_share[[
    "Model",
    "Dataset",
    "Depth",
    "Width",
    "Total Gates",
    "Optimized PBS",
    "PBS Gate Share After Reduction (%)",
]].sort_values(["Dataset", "Depth", "Width"])
pbs_gate_share_source.to_csv(OUT_DIR / "fps_optimized_pbs_gate_share_heatmap_source.csv", index=False)

share_min = pbs_gate_share_source["PBS Gate Share After Reduction (%)"].min()
share_max = pbs_gate_share_source["PBS Gate Share After Reduction (%)"].max()
share_mid = share_min + 0.55 * (share_max - share_min)

annotation_fontsize = PAPER_HEATMAP_ANNOTATION_FONTSIZE
tick_fontsize = PAPER_TICK_FONTSIZE
label_fontsize = PAPER_LABEL_FONTSIZE
title_fontsize = PAPER_TITLE_FONTSIZE

fig, axes = plt.subplots(1, len(DATASET_ORDER), figsize=(13.2, 4.8), sharey=True, constrained_layout=True)
for ax, dataset in zip(axes, DATASET_ORDER):
    group = pbs_gate_share_source[pbs_gate_share_source["Dataset"].astype(str) == dataset]
    heatmap = group.pivot(index="Depth", columns="Width", values="PBS Gate Share After Reduction (%)")
    heatmap = heatmap.sort_index().reindex(columns=sorted(heatmap.columns))

    image = ax.imshow(
        heatmap.to_numpy(),
        cmap="YlGnBu",
        vmin=share_min,
        vmax=share_max,
        aspect="auto",
    )
    ax.set_title(DATASET_LABELS[dataset], fontsize=title_fontsize, fontweight="bold", pad=10)
    ax.set_xlabel("Width (K gates/layer)", fontsize=label_fontsize, fontweight="bold", labelpad=8)
    ax.set_xticks(np.arange(len(heatmap.columns)))
    ax.set_xticklabels([f"{int(width / 1000)}" for width in heatmap.columns], fontsize=tick_fontsize, fontweight="bold")
    ax.set_yticks(np.arange(len(heatmap.index)))
    ax.set_yticklabels([int(depth) for depth in heatmap.index], fontsize=tick_fontsize, fontweight="bold")
    ax.tick_params(axis="both", width=PAPER_SPINE_WIDTH, length=5)
    for spine in ax.spines.values():
        spine.set_linewidth(PAPER_SPINE_WIDTH)
    ax.grid(False)

    for y, depth in enumerate(heatmap.index):
        for x, width in enumerate(heatmap.columns):
            value = heatmap.loc[depth, width]
            text_color = "white" if value >= share_mid else "black"
            ax.text(
                x,
                y,
                f"{value:.1f}",
                ha="center",
                va="center",
                color=text_color,
                fontsize=annotation_fontsize,
                fontweight="bold",
            )

axes[0].set_ylabel("Depth", fontsize=label_fontsize, fontweight="bold", labelpad=8)
colorbar = fig.colorbar(image, ax=axes, shrink=0.88, pad=0.02)
colorbar.set_label("Executed PBS share (%)", fontsize=label_fontsize, fontweight="bold", labelpad=10)
colorbar.ax.tick_params(labelsize=tick_fontsize, width=PAPER_SPINE_WIDTH, length=5)
for tick in colorbar.ax.get_yticklabels():
    tick.set_fontweight("bold")
colorbar.outline.set_linewidth(PAPER_SPINE_WIDTH)

for ext in ["pdf", "png"]:
    fig.savefig(OUT_DIR / f"fps_optimized_pbs_gate_share_heatmaps.{ext}", bbox_inches="tight")
plt.show()

display(pbs_gate_share_source.round({"PBS Gate Share After Reduction (%)": 2}))
print(OUT_DIR / "fps_optimized_pbs_gate_share_heatmaps.pdf")

## Figure 5: EI_DDLGN Encrypted Time vs. Width Curves

In [ ]:
ei_time_curves = ei_timing.copy()
ei_time_curves["width_k"] = ei_time_curves["width"] / 1000.0
ei_time_curves["dataset_label"] = ei_time_curves["dataset_name"].map(DATASET_LABELS)
ei_time_curves.to_csv(OUT_DIR / "fps_ei_encrypted_time_vs_width_source.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(13.8, 5.0), sharey=True, constrained_layout=True)
for ax, dataset in zip(axes, DATASET_ORDER):
    group = ei_time_curves[ei_time_curves["dataset_name"] == dataset]
    for depth, depth_group in group.groupby("depth"):
        depth_group = depth_group.sort_values("width")
        ax.plot(
            depth_group["width_k"],
            depth_group["ei_avg_eval_seconds"],
            marker="o",
            linewidth=PAPER_LINEWIDTH,
            markersize=PAPER_MARKERSIZE,
            label=f"D={int(depth)}",
        )
    ax.set_title(DATASET_LABELS[dataset], fontsize=PAPER_TITLE_FONTSIZE, fontweight="bold", pad=10)
    ax.set_xlabel("Width (K gates/layer)", fontsize=PAPER_LABEL_FONTSIZE, fontweight="bold", labelpad=8)
    ax.set_xticks([2, 4, 6, 8])
    ax.grid(True, linestyle="--", alpha=0.3)
    apply_paper_axis_style(ax)

axes[0].set_ylabel("EI_DDLGN encrypted eval. time\n(s/vector)", fontsize=PAPER_LABEL_FONTSIZE, fontweight="bold", labelpad=8)
legend = axes[-1].legend(title="Depth", loc="center left", bbox_to_anchor=(1.03, 0.5), frameon=True)
apply_paper_legend_style(legend)
#fig.suptitle("EI_DDLGN Encrypted Time vs. Width Curves", y=1.03)
#fig.tight_layout()

for ext in ["pdf", "png"]:
    fig.savefig(OUT_DIR / f"fps_ei_encrypted_time_vs_width.{ext}", bbox_inches="tight")
plt.show()
print(OUT_DIR / "fps_ei_encrypted_time_vs_width.pdf")

## Dataset-Level PBS Summary Table

In [ ]:
dataset_summary_for_paper = dataset_summary.copy()
dataset_summary_for_paper["Dataset"] = dataset_summary_for_paper["Dataset"].map(DATASET_LABELS)
dataset_summary_for_paper = dataset_summary_for_paper[[
    "Dataset",
    "Models",
    "Baseline PBS",
    "Optimized PBS",
    "PBS Saved",
    "Overall PBS Reduction %",
    "Mean Model PBS Reduction %",
    "max_pbs_reduction_percent",
]]
dataset_summary_for_paper = dataset_summary_for_paper.rename(columns={
    "Overall PBS Reduction %": "Overall red. (\\%)",
    "Mean Model PBS Reduction %": "Mean model red. (\\%)",
    "max_pbs_reduction_percent": "Max model red. (\\%)",
})
dataset_summary_for_paper[["Overall red. (\\%)", "Mean model red. (\\%)", "Max model red. (\\%)"]] = dataset_summary_for_paper[["Overall red. (\\%)", "Mean model red. (\\%)", "Max model red. (\\%)"]].round(2)

dataset_summary_for_paper.to_csv(OUT_DIR / "fps_dataset_pbs_summary_table.csv", index=False)
dataset_summary_latex = dataset_summary_for_paper.to_latex(
    index=False,
    escape=False,
    float_format="%.2f",
    caption="Dataset-level PBS reduction from public-constant propagation across all analyzed EI-DDLGN models.",
    label="tab:fps-dataset-pbs-summary",
    column_format="lrrrrrrr",
    position="t",
)
dataset_summary_latex = dataset_summary_latex.replace("\\begin{table}[t]\n", "\\begin{table}[t]\n\\centering\n")
(OUT_DIR / "fps_dataset_pbs_summary_table.tex").write_text(dataset_summary_latex, encoding="utf-8")
display(dataset_summary_for_paper)
print(OUT_DIR / "fps_dataset_pbs_summary_table.tex")

## Optional: Compact Comparison With Arithmetic TFHE Baselines

This table uses the reproduced QAT-FCNN baseline values from the previous manuscript and compares them with the three selected MNIST EI-DDLGN models. It reports absolute latency only; it does not report speedup or time reduction.

In [ ]:
if GENERATE_ARITHMETIC_COMPARISON:
    fcnn = pd.DataFrame([
        {"Model": "QAT-FCNN-4", "Size / active conn.": "56 active conn.", "Acc. (\\%)": 91.54, "Time/img (s)": 88.24, "Circuit bit-width": "6-bit: 50\\%, 7-bit: 50\\%"},
        {"Model": "QAT-FCNN-6", "Size / active conn.": "84 active conn.", "Acc. (\\%)": 92.25, "Time/img (s)": 126.97, "Circuit bit-width": "6-bit: 10\\%, 7-bit: 90\\%"},
        {"Model": "QAT-FCNN-8", "Size / active conn.": "112 active conn.", "Acc. (\\%)": 92.45, "Time/img (s)": 136.68, "Circuit bit-width": "7-bit: 100\\%"},
        {"Model": "QAT-FCNN-10", "Size / active conn.": "140 active conn.", "Acc. (\\%)": 92.29, "Time/img (s)": 135.98, "Circuit bit-width": "7-bit: 100\\%"},
        {"Model": "QAT-FCNN-11", "Size / active conn.": "154 active conn.", "Acc. (\\%)": 92.58, "Time/img (s)": 132.25, "Circuit bit-width": "7-bit: 100\\%"},
        {"Model": "QAT-FCNN-12", "Size / active conn.": "168 active conn.", "Acc. (\\%)": 92.42, "Time/img (s)": 207.98, "Circuit bit-width": "7-bit: 90\\%, 8-bit: 10\\%"},
    ])

    ei_mnist = selected_with_timing[selected_with_timing["Dataset"].astype(str) == "MNIST"].copy()
    ei_mnist["Model"] = ei_mnist.apply(lambda row: f"EI-DDLGN {row['Model Size']} (${int(row['D'])}\\times {row['W']}$)", axis=1)
    ei_mnist_comparison = pd.DataFrame({
        "Model": ei_mnist["Model"],
        "Size / active conn.": ei_mnist.apply(lambda row: f"{int(row['D'])} layers, {row['W']} width", axis=1),
        "Acc. (\\%)": ei_mnist["Accuracy (%)"].round(2),
        "Time/img (s)": ei_mnist["ei_avg_eval_seconds"].round(2),
        "Circuit bit-width": "Boolean gates",
    })

    arithmetic_comparison = pd.concat([ei_mnist_comparison, fcnn], ignore_index=True)
    arithmetic_comparison.to_csv(OUT_DIR / "fps_arithmetic_comparison_table.csv", index=False)
    arithmetic_latex = arithmetic_comparison.to_latex(
        index=False,
        escape=False,
        float_format="%.2f",
        caption="Compact MNIST comparison between EI-DDLGN and reproduced QAT-FCNN arithmetic TFHE baselines. Times are absolute per-image evaluation times.",
        label="tab:fps-arithmetic-comparison",
        column_format="llrrl",
        position="t",
    )
    arithmetic_latex = arithmetic_latex.replace("\\begin{table}[t]\n", "\\begin{table}[t]\n\\centering\n")
    (OUT_DIR / "fps_arithmetic_comparison_table.tex").write_text(arithmetic_latex, encoding="utf-8")
    display(arithmetic_comparison)
    print(OUT_DIR / "fps_arithmetic_comparison_table.tex")
else:
    print("Arithmetic comparison disabled.")

## Generated Artifacts

Main files to copy into the FPS paper:

- `fps_selected_models_table.tex` including `EI eval. (s)`
- `fps_accuracy_vs_optimized_pbs.pdf`
- `fps_pbs_reduction_distribution.pdf`
- `fps_best_accuracy_under_optimized_pbs_budget.pdf`
- `fps_optimized_pbs_gate_share_heatmaps.pdf`
- `fps_ei_encrypted_time_vs_width.pdf`
- `fps_dataset_pbs_summary_table.tex`
- `fps_arithmetic_comparison_table.tex` if the optional arithmetic comparison is enabled

All files are generated under `results/fps_paper_results/`.